## Outlier analysis
In this section, I will specifically filter the errors for the top 5% highest values per each type of error (i.e. rows of metrics). This is because these extreme outliers are harder to examine together with the regular data. After extracting these outliers, I will take the log of each of them, to make them smaller and more easy to handle. This also means that the concrete values of these errors and statistics of them are less meaningful in absolute value, but still inform us of the relative performance of hierarchies.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/base_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wweth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wfth)),
])

nrmse_em, nrmse_diffs = em, diffs

In [ ]:
nrmse_outliers_em = CustomMetrics.keep_outliers(nrmse_em, 0.95, log=True)

In [ ]:
nrmse_outliers_em.get_error_stats(MI(granularity=DAYS))

From the daily granularity errors we can see that the `Minimal` hierarchy's outlier statistics match the base forecast outliers' stats, each other hierarchy is farther off. The biggest difference is in the variance of these outliers, which is significantly the smallest in the case of `Minimal` hierarchy, with the smallest maximum also. From the way that the medians and means seem smaller for other hierarchies (compared to `Minimal` and base forecasts), but the variance and maximum is huge, we can deduct that those other hierarchies introduce some very extreme outliers by reconciliation, but overall reduce the 'volume' of outliers. The only percentile where we can see the other hierarchies worsen, is at the last p90 one.

It is hard to pinpoint a single best/worst performer here.

In [ ]:
nrmse_outliers_em.get_error_stats(MI(granularity=MONTHS))

Interestingly, in this case, `Minimal` hierarchy has the worst mean and percentile values (which are consistently worse than base forecasts as well, although no hierarchy is better than that in terms of these monthly outliers), but a slightly improved variance value. The difference in variance is not as large as with daily granularity. Because of all this, we can still suspect, that the other hierarchies produce a few but very large outliers.

At monthly granularity smallest reconciled percentiles and mean was produced by the `Semantic` hierarchy.

In [ ]:
nrmse_outliers_em.metrics.agg(lambda x: x.quantile(0.99), axis=1)

From manually checking the 99th percentile (so basically the 6th biggest value from 650 samples), we can still see that while Minimal is the smallest, other hierarchies still have not exploded yet (at least not significantly, mind that we are on log scale), meaning that our suspicion was correct and in fact there are very sparse and very large outliers in non `Minimal` hierarchies, which cause the big variances and max statistics.

Of course when we return to non-log scale, these outliers are very significant, and explode all the statistics, which we have seen did not really happen with `Minimal` hierarchy.

In [ ]:
CustomMetrics.keep_outliers(nrmse_em, 0.95, log=False).stats

Next we look at the histograms of the outlier distributions. Of course, they are still on log scale.

In [ ]:
plots.plot_error_distribution(nrmse_outliers_em,
                                MetricsIndex.indeces_from_product(granularities=[DAYS], hierarchies=[sth, mth, wth, wweth, wfth], forecast_types=[RECONCILED_FORECASTS]) + \
                                [MI(hierarchy=mth, granularity=DAYS, forecast_type=BASE_FORECASTS)],
                                title_info='Daily granularity error outliers',
                                bin_size=0.15,
                                outlier_threshold=0.999).show()

At daily distribution we can see that the outliers mostly decreased at each hierarchy by reconciliation. If we compare `Semantic` and `Minimal` specifically, then we can see that there are more smaller outliers with the `Semantic` hierarchy. We have already seen evidence for this from the statistics. Other hierarchies are similar to how `Semantic` looks.

In [ ]:
plots.plot_error_distribution(nrmse_outliers_em,
                                MetricsIndex.indeces_from_product(granularities=[MONTHS], hierarchies=[sth, mth, wth, wweth, wfth], forecast_types=[RECONCILED_FORECASTS]) + \
                                [MI(hierarchy=mth, granularity=MONTHS, forecast_type=BASE_FORECASTS)],
                                title_info='Monthly granularity error outliers',
                                bin_size=0.15,
                                outlier_threshold=0.999).show()

We can clearly see that with reconciliation, the outliers shifted in volume to being far greater, although their shape of distribution stayed more or less the same. If we compare `Semantic` and `Minimal`, we can see something similar which happened at the last diagram at daily granularity.

Mind that in both diagrams the biggest outliers were filtered out for clarity.

Next we look at box plots of outliers.

In [ ]:
plots.plot_error_boxes(nrmse_outliers_em,
                       MI(granularity=DAYS, forecast_type=RECONCILED_FORECASTS, hierarchy=[sth, mth, wth, wweth, wfth]))

Here in these plots we can clearly see the single huge outliers in all hierarchies, other than `Minimal`. If we zoom in to see just the box plots, they look very similar, with slightly shifted percentiles, as we have already seen in the statistics. The smaller outliers, outside of the 1.5 IQR range are quite uniform across all hierarchies.

### Conclusions
The top 5% highest errors (outliers) were extracted and log-transformed for analysis. At monthly granularity, the base forecast outliers were the smallest in terms of mean and percentiles — no reconciled hierarchy improved upon them. Extreme outliers that distort statistics (variance, max) were very rare across all hierarchies, though they did not appear at all with `Minimal`.

**Semantic**
- Pros
    - When looking at daily granularity reconciled forecast error outliers, this hierarchy had the best (lowest) median, across all reconciled hierarchies.
    - When looking at monthly granularity reconciled forecast error outliers, this hierarchy had the best mean and percentile values, across all reconciled hierarchies.
    - When looking at daily outlier histograms, this hierarchy shifted more outliers toward smaller values compared to the base forecast distribution.
- Cons
    - When looking at daily granularity reconciled forecast error outliers, this hierarchy introduced a handful of very extreme outliers through reconciliation, causing large variance and maximum values.

**Minimal**
- Pros
    - When looking at daily granularity reconciled forecast error outliers, this hierarchy's percentiles closely matched the base forecast outlier statistics, as also confirmed by the histogram distributions.
    - When looking at both daily and monthly granularity, this hierarchy introduced no extreme outliers through reconciliation, unlike all other hierarchies which had visible single huge outliers in the box plots.
    - When looking at daily granularity reconciled forecast error outliers, this hierarchy had the lowest variance, across all hierarchies.
    - When looking at monthly granularity reconciled forecast error outliers, this hierarchy had the lowest variance, across all hierarchies.
- Cons
    - When looking at monthly granularity reconciled forecast error outliers, this hierarchy had the worst (highest) mean and percentile values, across all reconciled hierarchies.

Highlights:
- Daily/Monthly outlier statistics